<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day2/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Comparaisons de l'attention multiple et du transformateur

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Configuration des graines aléatoires pour la reproductibilité
torch.manual_seed(42)

# =====================================================================
# TÂCHE 1 : IMPLÉMENTATION DE L'ATTENTION À UNE SEULE TÊTE (SCALED DOT-PRODUCT)
# =====================================================================
class ScaledDotProductAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        # Facteur d'échelle pour stabiliser les gradients (1 / sqrt(d_k))
        self.scale = 1.0 / math.sqrt(embed_dim)

    def forward(self, query, key, value, mask=None):
        # Étape A : Calcul des scores bruts par produit scalaire (Query x Key Transposée)
        # Formes : (batch, seq_len, seq_len)
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        # Application optionnelle du masque de remplissage (padding)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Étape B : Softmax pour convertir les scores en poids d'attention (somme = 1)
        attn_weights = F.softmax(scores, dim=-1)

        # Étape C : Multiplication des poids par les valeurs (Value)
        # Forme de sortie : (batch, seq_len, embed_dim)
        output = torch.matmul(attn_weights, value)

        return output, attn_weights

# =====================================================================
# TÂCHE 2 : MODULE D'ATTENTION MULTI-TÊTES (MULTI-HEAD ATTENTION)
# =====================================================================
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim doit être divisible par num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Projections linéaires pour Query, Key et Value
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)

        # Couche d'attention de base
        self.attention = ScaledDotProductAttention(self.head_dim)

        # Projection de sortie finale après concaténation
        self.out_linear = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size, seq_len, _ = query.size()

        # 1) Projections linéaires initiales
        q = self.q_linear(query)
        k = self.k_linear(key)
        v = self.v_linear(value)

        # 2) Séparation en plusieurs têtes (Split heads)
        # De (batch, seq_len, embed_dim) à (batch, num_heads, seq_len, head_dim)
        q = q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Ajustement du masque pour correspondre aux dimensions multi-têtes
        if mask is not None:
            mask = mask.unsqueeze(1) # (batch, 1, 1, seq_len)

        # 3) Calcul de l'attention par tête
        attn_output, attn_weights = self.attention(q, k, v, mask=mask)

        # 4) Concaténation des têtes (Concat heads)
        # Retour à la forme originale : (batch, seq_len, embed_dim)
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)

        # 5) Projection linéaire finale et Dropout
        output = self.out_linear(attn_output)
        return self.dropout(output), attn_weights

# =====================================================================
# TÂCHE 3 : PILE D'ENCODEURS PERSONNALISÉE (TRANSFORMER ENCODER BLOCK)
# =====================================================================
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        # Couche d'attention multi-têtes
        self.mha = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)

        # Réseau Feed-Forward (Réseau de propagation avant)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # --- Premier sous-bloc : Attention + Connexion résiduelle & Normalisation ---
        attn_out, weights = self.mha(x, x, x, mask=mask)
        x = self.norm1(x + attn_out) # Post-LN residual connection

        # --- Second sous-bloc : Feed-Forward + Connexion résiduelle & Normalisation ---
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x, weights

# =====================================================================
# VALIDATION DES FORMES ET VÉRIFICATION GÉOMÉTRIQUE (TEST)
# =====================================================================
print("--- Validation des dimensions de l'architecture personnalisée ---")

# Définition des dimensions de test (Simulant un lot de texte NLI)
BATCH_SIZE = 4
SEQ_LEN = 12     # Longueur de la séquence (Tokens)
EMBED_DIM = 64   # Dimension de l'embedding
NUM_HEADS = 4    # Nombre de têtes d'attention
FF_DIM = 256     # Dimension de la couche cachée Feed-Forward

# Création d'un tenseur d'entrée factice (batch, seq_len, hidden_dim)
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, EMBED_DIM)
print(f"Dimensions de l'entrée (Input Shape)  : {dummy_input.shape}")

# 1. Test du bloc d'attention multi-têtes seul
mha_layer = MultiHeadAttention(embed_dim=EMBED_DIM, num_heads=NUM_HEADS)
mha_out, attn_weights = mha_layer(dummy_input, dummy_input, dummy_input)

print("\n1. Validation du bloc Multi-Head Attention :")
print(f"   • Sortie MHA (Output Shape)        : {mha_out.shape}")
print(f"   • Cartes d'attention (Weights)     : {attn_weights.shape} -> (batch, heads, seq_len, seq_len)")

# 2. Test du bloc d'encodeur complet (Attention + Feed-Forward)
encoder_block = TransformerEncoderBlock(embed_dim=EMBED_DIM, num_heads=NUM_HEADS, ff_dim=FF_DIM)
encoder_out, final_weights = encoder_block(dummy_input)

print("\n2. Validation du bloc d'Encodeur complet :")
print(f"   • Sortie Encodeur (Output Shape)   : {encoder_out.shape}")
print(f"   • Validation stricte des formes    : {'✅ Réussie' if encoder_out.shape == dummy_input.shape else '❌ Échec'}")


--- Validation des dimensions de l'architecture personnalisée ---
Dimensions de l'entrée (Input Shape)  : torch.Size([4, 12, 64])

1. Validation du bloc Multi-Head Attention :
   • Sortie MHA (Output Shape)        : torch.Size([4, 12, 64])
   • Cartes d'attention (Weights)     : torch.Size([4, 4, 12, 12]) -> (batch, heads, seq_len, seq_len)

2. Validation du bloc d'Encodeur complet :
   • Sortie Encodeur (Output Shape)   : torch.Size([4, 12, 64])
   • Validation stricte des formes    : ✅ Réussie


1) Attention Mono-Tête vs Multi-Têtes : L'attention à tête unique calcule une unique moyenne pondérée sur tout l'espace d'embedding, ce qui force le modèle à se focaliser sur une seule relation sémantique dominante à la fois. L'attention multi-têtes divise la dimension d'embedding en sous-espaces distincts (head_dim), permettant au modèle de suivre en parallèle plusieurs relations syntaxiques indépendantes (par exemple, l'accord sujet-verbe dans une tête, et les dépendances pronominales à longue distance dans une autre).

2) Le rôle du facteur d'échelle (Scaling Factor) : Le produit scalaire entre Query et Key engendre des valeurs très élevées lorsque la dimension des vecteurs (embed_dim) grandit. Sans le facteur d'échelle \(\frac{1}{\sqrt{d_{k}}}\), la fonction Softmax se retrouve poussée vers des régions de forte saturation où ses dérivées mathématiques sont presque nulles. Diviser par la racine carrée de la dimension stabilise la variance à 1.0, évitant le phénomène de disparition du gradient (vanishing gradient) durant la rétropropagation.

3) Compromis d'architecture (Scratch vs Pre-trained) : Un encodeur d'attention personnalisé construit à partir de zéro (from scratch) possède l'avantage d'être extrêmement léger, d'offrir une vitesse d'inférence ultra-rapide et d'exiger très peu de mémoire vive, ce qui est parfait pour des tâches de classification ciblées. Cependant, il demande un volume massif de données pour converger. À l'inverse, un modèle lourd pré-entraîné (comme BERT ou DistilBERT) bénéficie de représentations linguistiques pré-acquises extrêmement riches, lui permettant d'atteindre des scores d'exactitude (Accuracy) bien supérieurs sur des tâches complexes comme l'inférence de langage naturel (NLI), mais au prix d'un coût de calcul (GPU) et d'une latence d'inférence nettement plus élevés.